# Reward Shaping SolarSystemLander

Colab L4 runner for ground thrust penalty reward shaping experiments.

## Set up

In [ ]:
# cell: colab-setup

!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
%pip install -r hpo/requirements.txt
%pip install -r reward_shaping/requirements.txt

import sys

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")
sys.path.insert(0, "reward_shaping/src")

from hpo.notebook.colab import setup_colab

COLAB = setup_colab()

In [ ]:
# cell: reward-shaping-setup; requires: colab-setup

from dataclasses import replace
import shutil

import torch

from hpo.solar_system_lander.environment import DEFAULT_WORLD_MIX, EnvFactory, World
from reward_shaping.experiment_harness import (
    ExperimentContext,
    run_experiment,
    vector_training_config_from_checkpoint,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

## Configure run

In [ ]:
# cell: run-config; requires: reward-shaping-setup

_OBSERVATION_MODE = "10d"
_SOURCE_EPS_END = 0.04400876385215351
EPS_START_VALUES = [_SOURCE_EPS_END, 2 * _SOURCE_EPS_END, 0.1, 0.2, 0.5, 1.0]

_ELISE_TRAINING_WORLD_MIX = {
    World.MERCURY: 1,
    World.VENUS: 4,
    World.EARTH: 4,
    World.MOON: 1,
    World.MARS: 1,
}
CONTEXT = ExperimentContext(
    source_storage_name=f"solar_system_lander_{_OBSERVATION_MODE}_elise_accel",
    drive_study_dir=COLAB.drive_study_dir,
    training_factory=EnvFactory(
        _OBSERVATION_MODE,
        world_mix=_ELISE_TRAINING_WORLD_MIX,
    ),
    evaluation_factory=EnvFactory(_OBSERVATION_MODE, world_mix=DEFAULT_WORLD_MIX),
    replay_memory_capacity=76_754,
    ground_thrust_penalty=0.5,
    num_envs=22,
    robust_episodes_per_world=50,
    training_seed=77,
    device=device,
)

# Keep the run close to the source checkpoint's HPO region.
# eps_start is swept because fine-tuning may need a small exploration bump.
BASE_TRAINING_CONFIG = vector_training_config_from_checkpoint(
    CONTEXT.initial_checkpoint,
    num_episodes=500,
    eps_start=_SOURCE_EPS_END,
    eps_end=_SOURCE_EPS_END,
)

## Run experiments

In [ ]:
# cell: run-experiments; requires: run-config

results = []

for eps_start in EPS_START_VALUES:
    training_config = replace(BASE_TRAINING_CONFIG, eps_start=eps_start)
    result = run_experiment(
        CONTEXT,
        training_config=training_config,
    )
    results.append(result)
    print(
        f"{result.run_id}: historical={result.historical_score.score:.1f}, "
        f"robust={result.robust_score.score:.1f}"
    )

## Save artifacts

In [ ]:
# cell: copy-runs-to-drive; requires: run-experiments

sweep_summary = []

for result in results:
    drive_run_path = CONTEXT.drive_run_root / result.run_id
    drive_run_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(result.paths.root, drive_run_path, dirs_exist_ok=True)
    sweep_summary.append(
        {
            "run_id": result.run_id,
            "historical_score": result.historical_score.score,
            "robust_score": result.robust_score.score,
            "historical_ground_side_thrust_steps": sum(
                row.ground_side_thrust_steps for row in result.historical_score.rows
            ),
            "robust_ground_side_thrust_steps": sum(
                row.ground_side_thrust_steps for row in result.robust_score.rows
            ),
            "drive_path": str(drive_run_path),
        }
    )

sweep_summary